# Option Metrics Calculator

This notebook loads **existing option quotes** and computes practical trading metrics such as moneyness, break-even, intrinsic/extrinsic value, annualized risk measures, probability proxy, and Black-Scholes Greeks.

## Expected input columns

Provide a CSV (or DataFrame) with at least the following columns:

- `symbol`
- `option_type` (`call` or `put`)
- `strike`
- `expiry` (YYYY-MM-DD)
- `underlying_price`
- `option_price` (mid/last/theoretical, your choice)
- `implied_vol` (decimal form, e.g. 0.25)

Optional columns:
- `risk_free_rate` (decimal; defaults to 0.04 if omitted)
- `contracts` (position size; defaults to 1)

In [1]:
import math
from datetime import datetime, timezone

import numpy as np
import pandas as pd

In [5]:
# --- Configure your data source ---
CSV_PATH = 'C:\\Users\\alezu\\Documents\\Projects\\option-risk\\data\\option_data.csv'  # Update this path to your option quote file
DEFAULT_RF = 0.04  # Fallback risk-free rate when not provided

In [6]:
df = pd.read_csv(CSV_PATH)
df['symbol'] = "MSFT"
df['option_type'] = df['option_type'].apply(lambda x: 'call' if x else 'put')
df.head()

,symbol,option_type,strike,implied_vol,option_price,bid,ask,expiry,inTheMoney,days_to_expiry,underlying_price
0,MSFT,call,180,0.735720,296.90,324.50,329.5,18/12/2026,True,406,497.100006
1,MSFT,call,190,0.891603,340.52,326.35,331.0,18/12/2026,True,406,497.100006
2,MSFT,call,195,0.877870,336.02,321.85,326.5,18/12/2026,True,406,497.100006
3,MSFT,call,210,0.827059,324.77,307.65,312.0,18/12/2026,True,406,497.100006
4,MSFT,call,225,0.781679,312.60,293.25,298.0,18/12/2026,True,406,497.100006


In [7]:
required_cols = {
    'symbol', 'option_type', 'strike', 'expiry', 'underlying_price',
    'option_price', 'implied_vol'
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

df = df.copy()
df['option_type'] = df['option_type'].str.lower().str.strip()
df['expiry'] = pd.to_datetime(df['expiry'], utc=True, errors='coerce')
if df['expiry'].isna().any():
    bad_idx = df.index[df['expiry'].isna()].tolist()
    raise ValueError(f'Invalid expiry dates at rows: {bad_idx}')

for col in ['strike', 'underlying_price', 'option_price', 'implied_vol']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

if 'risk_free_rate' not in df.columns:
    df['risk_free_rate'] = DEFAULT_RF
else:
    df['risk_free_rate'] = pd.to_numeric(df['risk_free_rate'], errors='coerce').fillna(DEFAULT_RF)

if 'contracts' not in df.columns:
    df['contracts'] = 1
else:
    df['contracts'] = pd.to_numeric(df['contracts'], errors='coerce').fillna(1).clip(lower=0)

if df[['strike', 'underlying_price', 'option_price', 'implied_vol']].isna().any().any():
    raise ValueError('Numeric columns contain invalid values after parsing.')

df.head()

C:\Users\alezu\AppData\Local\Temp\ipykernel_16308\1577425318.py:11: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['expiry'] = pd.to_datetime(df['expiry'], utc=True, errors='coerce')


,symbol,option_type,strike,implied_vol,option_price,bid,ask,expiry,inTheMoney,days_to_expiry,underlying_price,risk_free_rate,contracts
0,MSFT,call,180,0.735720,296.90,324.50,329.5,2026-12-18 00:00:00+00:00,True,406,497.100006,0.04,1
1,MSFT,call,190,0.891603,340.52,326.35,331.0,2026-12-18 00:00:00+00:00,True,406,497.100006,0.04,1
2,MSFT,call,195,0.877870,336.02,321.85,326.5,2026-12-18 00:00:00+00:00,True,406,497.100006,0.04,1
3,MSFT,call,210,0.827059,324.77,307.65,312.0,2026-12-18 00:00:00+00:00,True,406,497.100006,0.04,1
4,MSFT,call,225,0.781679,312.60,293.25,298.0,2026-12-18 00:00:00+00:00,True,406,497.100006,0.04,1


In [8]:
def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def norm_pdf(x: float) -> float:
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)


def bs_greeks(option_type: str, s: float, k: float, t: float, r: float, sigma: float):
    # Handle near-zero time or vol to avoid unstable division
    t = max(t, 1e-8)
    sigma = max(sigma, 1e-8)

    d1 = (math.log(s / k) + (r + 0.5 * sigma**2) * t) / (sigma * math.sqrt(t))
    d2 = d1 - sigma * math.sqrt(t)

    if option_type == 'call':
        delta = norm_cdf(d1)
        theta = (
            -(s * norm_pdf(d1) * sigma) / (2 * math.sqrt(t))
            - r * k * math.exp(-r * t) * norm_cdf(d2)
        )
        rho = k * t * math.exp(-r * t) * norm_cdf(d2)
    elif option_type == 'put':
        delta = norm_cdf(d1) - 1
        theta = (
            -(s * norm_pdf(d1) * sigma) / (2 * math.sqrt(t))
            + r * k * math.exp(-r * t) * norm_cdf(-d2)
        )
        rho = -k * t * math.exp(-r * t) * norm_cdf(-d2)
    else:
        raise ValueError(f'Unknown option_type: {option_type}')

    gamma = norm_pdf(d1) / (s * sigma * math.sqrt(t))
    vega = s * norm_pdf(d1) * math.sqrt(t)

    # theta returned in annualized terms; trading desks often use theta/day
    theta_per_day = theta / 365.0

    return {
        'd1': d1,
        'd2': d2,
        'delta': delta,
        'gamma': gamma,
        'vega': vega / 100.0,      # per +1 vol point
        'theta_day': theta_per_day,
        'rho': rho / 100.0         # per +1% rate move
    }

In [9]:
now = pd.Timestamp(datetime.now(timezone.utc))
df['days_to_expiry'] = (df['expiry'] - now).dt.total_seconds() / 86400.0
df['days_to_expiry'] = df['days_to_expiry'].clip(lower=0)
df['t_years'] = (df['days_to_expiry'] / 365.0).clip(lower=1e-8)

is_call = df['option_type'] == 'call'
is_put = df['option_type'] == 'put'

if (~(is_call | is_put)).any():
    bad_types = df.loc[~(is_call | is_put), 'option_type'].unique().tolist()
    raise ValueError(f'option_type must be call/put only. Found: {bad_types}')

# Intrinsic / Extrinsic
df['intrinsic'] = np.where(is_call,
                           np.maximum(df['underlying_price'] - df['strike'], 0.0),
                           np.maximum(df['strike'] - df['underlying_price'], 0.0))
df['extrinsic'] = df['option_price'] - df['intrinsic']

# Moneyness & break-even
df['moneyness_spot_over_strike'] = df['underlying_price'] / df['strike']
df['break_even'] = np.where(is_call,
                            df['strike'] + df['option_price'],
                            df['strike'] - df['option_price'])

# Exposure
df['max_loss_long'] = df['option_price'] * 100 * df['contracts']
df['notional_underlying'] = df['underlying_price'] * 100 * df['contracts']
df['premium_pct_of_underlying'] = (df['option_price'] / df['underlying_price']) * 100

# N(d2) style probability proxy (risk-neutral ITM probability under BS assumptions)
probs = []
greek_rows = []
for row in df.itertuples(index=False):
    greeks = bs_greeks(
        option_type=row.option_type,
        s=float(row.underlying_price),
        k=float(row.strike),
        t=float(row.t_years),
        r=float(row.risk_free_rate),
        sigma=float(row.implied_vol),
    )
    greek_rows.append(greeks)
    probs.append(norm_cdf(greeks['d2']) if row.option_type == 'call' else norm_cdf(-greeks['d2']))

greeks_df = pd.DataFrame(greek_rows)
df = pd.concat([df.reset_index(drop=True), greeks_df.reset_index(drop=True)], axis=1)
df['prob_itm_proxy'] = probs

# Position Greeks (per contract multiplier = 100)
for g in ['delta', 'gamma', 'vega', 'theta_day', 'rho']:
    df[f'position_{g}'] = df[g] * 100 * df['contracts']

metrics_cols = [
    'symbol', 'option_type', 'expiry', 'strike', 'underlying_price', 'option_price',
    'days_to_expiry', 'implied_vol', 'intrinsic', 'extrinsic',
    'moneyness_spot_over_strike', 'break_even', 'premium_pct_of_underlying',
    'prob_itm_proxy', 'delta', 'gamma', 'vega', 'theta_day', 'rho',
    'position_delta', 'position_gamma', 'position_vega', 'position_theta_day', 'position_rho'
]

result = df[metrics_cols].sort_values(['symbol', 'expiry', 'strike']).reset_index(drop=True)
result.head(20)

,symbol,option_type,expiry,strike,underlying_price,option_price,days_to_expiry,implied_vol,intrinsic,extrinsic,...,delta,gamma,vega,theta_day,rho,position_delta,position_gamma,position_vega,position_theta_day,position_rho
0,MSFT,call,2026-12-18 00:00:00+00:00,175,497.100006,0.37,246.626112,0.250007,322.100006,-321.730006,...,1.000000,2.875661e-09,1.200396e-06,-0.018667,1.150923,99.999995,2.875661e-07,1.200396e-04,-1.866675,115.092300
1,MSFT,call,2026-12-18 00:00:00+00:00,180,497.100006,296.90,246.626112,0.735720,317.100006,-20.200006,...,0.978658,1.701622e-04,2.090305e-01,-0.048890,1.092051,97.865823,1.701622e-02,2.090305e+01,-4.889022,109.205121
2,MSFT,call,2026-12-18 00:00:00+00:00,180,497.100006,0.27,246.626112,0.125009,317.100006,-316.830006,...,1.000000,1.995296e-25,4.164679e-23,-0.019200,1.183807,100.000000,1.995296e-23,4.164679e-21,-1.920002,118.380670
3,MSFT,call,2026-12-18 00:00:00+00:00,185,497.100006,0.38,246.626112,0.125009,312.100006,-311.720006,...,1.000000,2.922380e-24,6.099733e-22,-0.019733,1.216690,100.000000,2.922380e-22,6.099733e-20,-1.973336,121.669022
4,MSFT,call,2026-12-18 00:00:00+00:00,190,497.100006,340.52,246.626112,0.891603,307.100006,33.419994,...,0.956883,2.513581e-04,3.741949e-01,-0.084605,1.046048,95.688275,2.513581e-02,3.741949e+01,-8.460520,104.604767
5,MSFT,call,2026-12-18 00:00:00+00:00,190,497.100006,0.44,246.626112,0.125009,307.100006,-306.660006,...,1.000000,3.721552e-23,7.767803e-21,-0.020267,1.249574,100.000000,3.721552e-21,7.767803e-19,-2.026669,124.957374
6,MSFT,call,2026-12-18 00:00:00+00:00,195,497.100006,336.02,246.626112,0.877870,302.100006,33.919994,...,0.954969,2.643888e-04,3.875314e-01,-0.086336,1.070643,95.496881,2.643888e-02,3.875314e+01,-8.633586,107.064282
7,MSFT,call,2026-12-18 00:00:00+00:00,195,497.100006,0.41,246.626112,0.125009,302.100006,-301.690006,...,1.000000,4.158012e-22,8.678804e-20,-0.020800,1.282457,100.000000,4.158012e-20,8.678804e-18,-2.080002,128.245726
8,MSFT,call,2026-12-18 00:00:00+00:00,200,497.100006,0.46,246.626112,0.125009,297.100006,-296.640006,...,1.000000,4.109628e-21,8.577814e-19,-0.021333,1.315341,100.000000,4.109628e-19,8.577814e-17,-2.133336,131.534078
9,MSFT,call,2026-12-18 00:00:00+00:00,205,497.100006,0.61,246.626112,0.125009,292.100006,-291.490006,...,1.000000,3.620391e-20,7.556655e-18,-0.021867,1.348224,100.000000,3.620391e-18,7.556655e-16,-2.186669,134.822430


In [10]:
# Portfolio-level snapshot
portfolio = {
    'total_contracts': float(df['contracts'].sum()),
    'total_premium_at_risk': float(df['max_loss_long'].sum()),
    'net_delta': float(df['position_delta'].sum()),
    'net_gamma': float(df['position_gamma'].sum()),
    'net_vega_per_vol_pt': float(df['position_vega'].sum()),
    'net_theta_per_day': float(df['position_theta_day'].sum()),
    'net_rho_per_1pct_rate': float(df['position_rho'].sum()),
}
pd.Series(portfolio).to_frame('value')

,value
total_contracts,184.000000
total_premium_at_risk,804043.000000
net_delta,9275.109939
net_gamma,9.191633
net_vega_per_vol_pt,1395.014601
net_theta_per_day,-412.910475
net_rho_per_1pct_rate,26785.972224


In [ ]:
# Save results for downstream workflows
OUTPUT_PATH = '../data/option_metrics_output.csv'
result.to_csv(OUTPUT_PATH, index=False)
print(f'Wrote metrics to: {OUTPUT_PATH}')